# The hottest topics in machine learning: LDA over NIPS paper titles

**Status:** ✅ Runs end to end — falls back to a **clearly-labelled synthetic** corpus when the real NIPS archive is absent

**Demonstrates:** Unsupervised topic modelling · text preprocessing · bag-of-words. The 110 MB unreassembled multi-part download this depended on is gone; `fetch_data.py --only lda` explains how to get the real thing.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


## What was broken

1. **The dataset was never usable.** The repository contained
   `papers.csv.gz.part_aa` through `.part_ah` - a multi-part download that was
   never reassembled. Those fragments are deleted; `scripts/fetch_data.py`
   provides a working source. If the corpus is absent this notebook falls back
   to a small **synthetic** corpus, clearly labelled as such, so the code path
   still runs.
2. **`get_feature_names()` was removed in scikit-learn 1.0.** Both helper
   functions used it; `get_feature_names_out()` is the replacement.
3. **`re.sub('[,\.!?]', '', x)` was not the cleaning it looked like.** That
   character class removes only four punctuation marks, and raises `TypeError`
   on a missing title.
4. **`wordcloud.to_image()` does not display in Jupyter** - it returns a PIL
   image that has to be passed to `display()` or plotted.

## 1. Load the corpus

In [ ]:
import pandas as pd
import re

# The NIPS paper corpus is ~28 MB and is therefore not committed to git.
# Fetch it with:  python ../../scripts/fetch_data.py --only lda
#
# If it is absent, a small synthetic stand-in is built so the rest of the
# notebook still runs. It is clearly labelled as synthetic wherever it is
# used, because topics discovered in invented text say nothing about the field.
PAPERS_PATH = "../../data/nlp/papers.csv.gz"

import os


SYNTHETIC = False
if os.path.exists(PAPERS_PATH):
    papers = pd.read_csv(PAPERS_PATH)
    SYNTHETIC = False
else:
    print(f"! {PAPERS_PATH} not found - building a SYNTHETIC stand-in.")
    print("! Run `python ../../scripts/fetch_data.py --only lda` for the real corpus.")
    print("! Topics found below describe invented text, not the literature.")
    import random

    random.seed(0)
    vocab = {
        "neural": ["network", "deep", "learning", "gradient"],
        "bayesian": ["inference", "posterior", "prior", "markov"],
        "vision": ["image", "recognition", "kernel", "descriptor"],
        "optimisation": ["convex", "gradient", "convergence", "regularisation"],
        "graph": ["network", "graph", "embedding", "node"],
    }
    all_words = [word for words in vocab.values() for word in words]
    rows = []
    for index in range(2400):
        topic = random.choice(list(vocab))
        # Two words from the document's own topic, one from a neighbour, so the
        # topics overlap slightly the way real research areas do.
        neighbours = random.sample([t for t in vocab if t != topic], 1)
        title = " ".join(random.sample(vocab[topic], 3) + random.sample(vocab[neighbours[0]], 2))
        filler = " ".join(random.sample(all_words, 3))
        rows.append({
            "id": index,
            "title": f"{title} {filler}",
            "year": random.randint(1987, 2016),
            "event_type": "conference",
            "pdf_name": f"paper{index}.pdf",
        })
    papers = pd.DataFrame(rows)
    SYNTHETIC = True

print("synthetic data:" , SYNTHETIC, "| rows:", len(papers))
print(papers.head())


## 2. How the field grew

In [ ]:
import matplotlib.pyplot as plt

# How many papers per year? Machine learning as a field grew enormously over
# this period, which is the context for the topic analysis that follows.
counts = papers.groupby("year").size()

plt.figure(figsize=(9, 4))
plt.bar(counts.index, counts.values, color="tab:blue", width=0.8)
plt.xlabel("year")
plt.ylabel("number of papers")
plt.title("NIPS submissions per year" + ("  (SYNTHETIC DATA)" if SYNTHETIC else ""))
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"total papers: {len(papers)}")
print(f"year range: {counts.index.min()} - {counts.index.max()}")


## 3. Clean the titles

In [ ]:
# Clean the titles.
#
# BUGFIX: the original used re.sub('[,\\.!?]', '', x), a character class that
# removes only commas, full stops, exclamation and question marks, and then
# called .str.lower() on the result. Any other punctuation survives, and a NaN
# title raises a TypeError inside re.sub. Unicode-aware regex + an explicit
# null guard fixes both.
PUNCTUATION = re.compile(r"[^a-z0-9\s]")


def clean_title(title):
    """Normalise a paper title for bag-of-words analysis.

    Args:
        title: Raw title string, possibly ``None`` or ``NaN``.

    Returns:
        str: Lowercase title with punctuation removed and whitespace collapsed.
        An empty string if the input is missing or contains no word characters.

    Example:
        >>> clean_title("Deep Learning, 2016!")
        'deep learning 2016'
        >>> clean_title(None)
        ''
    """
    if not isinstance(title, str):
        return ""
    without_punctuation = PUNCTUATION.sub(" ", title.lower())
    return " ".join(without_punctuation.split())


papers["title_processed"] = papers["title"].map(clean_title)
empty = int((papers["title_processed"] == "").sum())
print(f"titles with no usable characters after cleaning: {empty}")
print(papers["title_processed"].head())


## 4. Check the cleaning visually

In [ ]:
from wordcloud import WordCloud
from IPython.display import display

# A word cloud is a quick visual check that the cleaning worked: if stray
# punctuation or template text is dominating, it shows up here immediately.
long_string = papers["title_processed"].str.cat(sep=" ")

wordcloud = WordCloud(
    width=1200, height=600, background_color="white", max_words=100,
    collocations=False,
)
wordcloud.generate(long_string)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word cloud of paper titles"
          + ("  (SYNTHETIC DATA)" if SYNTHETIC else ""), fontsize=14)
plt.tight_layout()
plt.show()


## 5. Bag of words

`min_df=2` drops terms appearing in fewer than two documents (typos, citations,
one-off model names). `max_df=0.5` drops terms in more than half the corpus
(template boilerplate). Neither bound changes the maths, but both shrink the
vocabulary and make the topics readable.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Drop words that appear in fewer than 2 documents: they are almost always
# typos, model names or citations, and they inflate the vocabulary without
# carrying topical signal.
# BUGFIX: the original passed `stop_words=ENGLISH_STOP_WORDS`, importing the
# frozen set from sklearn.feature_extraction.text. From scikit-learn 1.2 that
# import raises InvalidParameterError - `stop_words` now takes the string
# "english" and scikit-loads the list itself.
count_vectorizer = CountVectorizer(
    stop_words="english",
    min_df=2,
    max_df=0.5,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",   # letters only, min length 2
)
count_data = count_vectorizer.fit_transform(papers["title_processed"])
print("document-term matrix:", count_data.shape)
print("vocabulary size:", len(count_vectorizer.vocabulary_))


def plot_most_common_words(count_data, vectorizer, n=10):
    """Plot the n most frequent terms in a document-term matrix.

    BUGFIX: the original used ``vectorizer.get_feature_names()``, removed in
    scikit-learn 1.0 in favour of ``get_feature_names_out()``.

    Args:
        count_data: Sparse document-term matrix of shape
            ``(n_documents, n_terms)``.
        vectorizer: The fitted :class:`~sklearn.feature_extraction.text.CountVectorizer`
            that produced it.
        n: How many terms to show.

    Returns:
        list[tuple[str, float]]: The (term, count) pairs, most frequent first.
    """
    terms = vectorizer.get_feature_names_out()
    totals = np.asarray(count_data.sum(axis=0)).ravel()
    pairs = sorted(zip(terms, totals), key=lambda pair: pair[1], reverse=True)[:n]
    words = [w for w, _ in pairs]
    counts = [c for _, c in pairs]

    plt.figure(figsize=(9, 4))
    plt.bar(words, counts, color="tab:orange")
    plt.xticks(rotation=90)
    plt.xlabel("term")
    plt.ylabel("count")
    plt.title(f"{n} most common terms")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    return pairs


top_terms = plot_most_common_words(count_data, count_vectorizer, n=10)


## 6. Latent Dirichlet Allocation

Each document gets a distribution over `N_TOPICS` latent topics; each topic
gets a distribution over words. The fitted `components_` matrix *is* the
topic-by-word distribution, so the top words per topic are the model's own
output rather than something added afterwards.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# LDA models each document as a distribution over topics, and each topic as a
# distribution over words. It is generative: you can sample a document by
# picking a topic, then a word from that topic. That explicit structure is why
# the resulting topics are directly readable from their top words.
N_TOPICS = 14
N_TOP_WORDS = 12

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=0,
    learning_method="batch",
    max_iter=20,
)
lda.fit(count_data)


def print_topics(model, vectorizer, n_top_words=N_TOP_WORDS):
    """Print the most probable words for each topic.

    BUGFIX: the original called ``vectorizer.get_feature_names()``, removed in
    scikit-learn 1.0.

    Args:
        model: A fitted :class:`~sklearn.decomposition.LatentDirichletAllocation`.
        vectorizer: The fitted vectorizer defining the term ordering.
        n_top_words: Words to show per topic.

    Returns:
        list[list[str]]: The top words for each topic, in topic order.
    """
    terms = vectorizer.get_feature_names_out()
    topics = []
    for topic_index, topic_weights in enumerate(model.components_):
        top_indices = topic_weights.argsort()[::-1][:n_top_words]
        words = [terms[i] for i in top_indices]
        topics.append(words)
        print(f"Topic {topic_index:2d}: {' '.join(words)}")
    return topics


print(f"topics found via LDA ({N_TOPICS} topics, top {N_TOP_WORDS} words each):")
topics = print_topics(lda, count_vectorizer)
print()
if SYNTHETIC:
    print("! These topics describe SYNTHETIC text. Fetch the real corpus to see")
    print("! what the model actually finds in the NIPS literature.")


## 7. The full topic-term matrix

In [ ]:
# A topic-by-term heat map: the raw component matrix, before reducing each row
# to its top words. It shows how sharply peaked each topic is - a diffuse row
# means the topic overlaps with its neighbours.
components = lda.components_
terms = count_vectorizer.get_feature_names_out()

order = np.argsort(-components.max(axis=1))          # most peaked topics first
components = components[order]

plt.figure(figsize=(12, max(4, N_TOPICS * 0.32)))
plt.imshow(components, aspect="auto", cmap="magma", interpolation="nearest")
plt.colorbar(label="word weight in topic")
plt.xlabel("term index")
plt.ylabel("topic (sorted by peak weight)")
plt.title("LDA topic-term weight matrix"
          + ("  (SYNTHETIC DATA)" if SYNTHETIC else ""))
plt.tight_layout()
plt.show()
